# Stage1 10:32 恢复动作：首小时有方向成交量来源

只构造来源和时钟，不读取净动作标签、模型或账户；完整范围与停止门见 plan.json。

In [1]:
import hashlib
import json
import os
import shlex
from pathlib import Path

import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R
from src.data.minute_intake import atomic_json
from src.data.nas_market_data import _ssh, archive_nas_recorder

os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
root = Path.cwd().resolve()
card = root / 'research/stage1_intraday_signed_volume_action_source_20261023'
plan = json.loads((card / 'plan.json').read_text())
source = plan['source']
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert plan['frozen'] and plan['budget'] == {'new_market_requests':0,'Qlib_D_feature_reads':1,'source_Recorder_object_loads':0,'market_outcome_label_reads':0,'model_fits':0,'native_accounts':0,'new_Qlib_recorders':1}
assert not (card / 'run_state.json').exists()
assert sha(root / source['provider_uri'] / 'qlib_store_manifest.json') == source['provider_manifest_sha256']
manifest = json.loads((root / source['provider_uri'] / 'qlib_store_manifest.json').read_text())
assert manifest['source'] == 'NAS raw NONE snapshot sha256=' + source['source_raw_NONE_snapshot_sha256']
readback = json.loads((root / source['source_NAS_readback']).read_text())
assert readback['NAS_archive_sha256'] == source['source_NAS_archive_sha256'] and readback['artifact_hashes_checked'] == 118
binding = json.loads((root / source['source_card'] / 'nas_binding.json').read_text())
assert binding['verified'] and binding['evidence']['sha256'] == source['source_NAS_archive_sha256']
from src.workflow.intraday_recovery_direct_allocation import FEATURE_COLUMNS
assert not any('signed_volume' in item for item in FEATURE_COLUMNS)

def signed_pressure(close, volume):
    close=np.asarray(close,float);volume=np.asarray(volume,float)
    assert len(close)==len(volume)==60 and np.isfinite(close).all() and np.isfinite(volume).all() and (close>0).all() and (volume>=0).all()
    denom=volume[1:].sum()
    return float((np.sign(np.log(close[1:]/close[:-1]))*volume[1:]).sum()/denom) if denom>0 else np.nan
assert signed_pressure(np.arange(1,61),np.ones(60)) == 1.0
assert signed_pressure(np.arange(60,0,-1),np.ones(60)) == -1.0
qlib.init(provider_uri=str(root / source['provider_uri']),region='us')
cal=pd.DatetimeIndex(D.calendar(start_time=plan['calendar']['training_action_cohort'][0],end_time=plan['calendar']['reused_held_action_cohort'][1],freq='day'))
train=cal[(cal>=pd.Timestamp('2018-10-15'))&(cal<=pd.Timestamp('2022-12-30'))]
held=cal[(cal>=pd.Timestamp('2023-01-04'))&(cal<=pd.Timestamp('2026-10-08'))]
assert len(train)==plan['calendar']['expected_training_dates'] and len(held)==plan['calendar']['expected_held_dates']
fields=['$close','$volume','$price_valid']
raw=D.features(['SOXX'],fields,start_time='2018-10-15 09:31:00',end_time='2026-10-08 16:00:00',freq='1min').reset_index()
assert list(raw.columns)==['instrument','datetime',*fields] and raw.instrument.eq('SOXX').all()
raw['date']=pd.to_datetime(raw.datetime).dt.normalize()
bydate={d:g.set_index('datetime') for d,g in raw.groupby('date',sort=False)}
rows=[]
for day in cal:
    expected=pd.date_range(day+pd.Timedelta(hours=9,minutes=31),periods=60,freq='min')
    group=bydate.get(day)
    if group is None:
        rows.append({'trade_date':day,'source_bars_first_hour':0,'invalid_bars':0,'pressure':np.nan,'status':'missing_session'});continue
    first=group.reindex(expected)
    invalid=(~np.isfinite(first[fields].to_numpy(float))).any(axis=1) | first['$close'].le(0).to_numpy() | first['$volume'].lt(0).to_numpy() | first['$price_valid'].ne(1).to_numpy()
    count=int(first['$close'].notna().sum())
    if count!=60:
        status='missing_first_hour_bar';value=np.nan
    elif invalid.any():
        status='invalid_first_hour_bar';value=np.nan
    elif first['$volume'].to_numpy(float)[1:].sum()<=0:
        status='zero_signed_volume_denominator';value=np.nan
    else:
        status='known';value=signed_pressure(first['$close'],first['$volume'])
    rows.append({'trade_date':day,'source_bars_first_hour':count,'invalid_bars':int(invalid.sum()),'pressure':value,'status':status})
daily=pd.DataFrame(rows)
daily['assumed_available_at']=daily.trade_date+pd.Timedelta(hours=10,minutes=31)
daily['receiver_decision_at']=daily.trade_date+pd.Timedelta(hours=10,minutes=32)
assert (daily.assumed_available_at<daily.receiver_decision_at).all()
assert daily.trade_date.is_unique and len(daily)==len(cal)
training=daily[daily.trade_date.isin(train)]
held_frame=daily[daily.trade_date.isin(held)]
known_train=int(training.status.eq('known').sum());known_held=int(held_frame.status.eq('known').sum())
fractions={'training':known_train/len(train),'held':known_held/len(held)}
pass_gate=all(x>=plan['calendar']['minimum_known_fraction_each_cohort'] for x in fractions.values())
state={'status':'source_constructed','plan_sha256':sha(card/'plan.json'),'Qlib_D_feature_reads':1,'new_market_requests':0,'market_outcome_label_reads':0,'model_fits':0,'native_accounts':0,'new_Qlib_recorders':0}
atomic_json(state,card/'run_state.json')
summary={'experiment':plan['experiment'],'decision':'conditional_source_gate_passed_first_receipt_unqualified_no_fit' if pass_gate else 'needs_source_quality_no_fit','objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'source_provider_manifest_sha256':source['provider_manifest_sha256'],'source_NAS_archive_sha256':source['source_NAS_archive_sha256'],'source_rows':len(raw),'source_calendar_days':len(cal),'training_calendar_days':len(train),'training_known_days':known_train,'held_calendar_days':len(held),'held_known_days':known_held,'known_fractions':fractions,'source_gate_passed':pass_gate,'unknown_training_dates':[str(x.date()) for x in training.loc[training.status.ne('known'),'trade_date']],'unknown_held_dates':[str(x.date()) for x in held_frame.loc[held_frame.status.ne('known'),'trade_date']],'status_counts':{str(k):int(v) for k,v in daily.status.value_counts().items()},'first_receipt_and_finality_qualified':False,'new_market_requests':0,'market_outcome_label_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/daily','analysis/source_first_hour']}}
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
    recorder=R.get_recorder()
    recorder.save_objects(artifact_path='inputs',plan=plan,provider_manifest=manifest,source_NAS_readback=readback)
    recorder.save_objects(artifact_path='analysis',daily=daily,source_first_hour=raw[['instrument','datetime','date',*fields]])
    recorder.log_metrics(model_fits=0,native_accounts=0,training_known_fraction=fractions['training'],held_known_fraction=fractions['held'])
    summary.update(recorder_id=recorder.id,experiment_id=str(recorder.experiment_id),recorder_uri=str(card/'qlib_recorder'))
atomic_json(summary,card/'summary.json')
new_binding=json.loads((root / source['source_card'] / 'nas_binding.json').read_text())
new_binding.pop('evidence',None)
new_binding.update(source_id=plan['experiment'],source_plan='plan.json',as_of='2026-10-09')
atomic_json(new_binding,card/'nas_binding.json')
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=summary['recorder_id'],experiment_id=summary['experiment_id'])
remote=_ssh(new_binding['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0]
assert remote==evidence['sha256']
summary['nas_evidence']=evidence
summary['nas_readback']={'sha256':remote,'bytes':evidence['size_bytes'],'verified':True}
atomic_json(summary,card/'summary.json')
state.update(status='completed',new_Qlib_recorders=1,recorder_id=summary['recorder_id'])
atomic_json(state,card/'run_state.json')
(card/'conclusion.md').write_text(
    '# Stage1 10:32 恢复动作：首小时带方向成交量来源\n\n'
    f'只从既有NAS原始NONE来源的Qlib SOXX 1分钟库读取完成的09:31–10:30共60根bar，按冻结公式得到正负方向成交量比。训练动作日历2018-10-15至2022-12-30为{len(train)}日，其中{known_train}日完整已知；复用评价动作日历2023-01-04至2026-10-08为{len(held)}日，其中{known_held}日完整已知。所有未知日保留，未插值、过滤压力日或改时间切点。\n\n'
    f'来源门`{summary["decision"]}`。该坐标在研究口径10:31可构造，晚于旧09:29配置决策，但早于10:32恢复动作；它没有补齐2011–2018日频配置头的分钟训练缺口。旧恢复模型已有首小时收益、振幅和总量，没有此时段的方向加权量。实际历史首收、bar最终性与复权版本尚不合格；本卡未读取净动作优势标签、SOXX未来结果，0拟合、0账户。\n\n'
    f'下一步仅在来源门通过时，先冻结与旧恢复21项输入匹配、实际beta0.6且标签成熟的净恢复/HOLD局部信息检验；若失败即关闭，不依据结果换符号或cutoff。局部信息通过仍须原Stage1完整0bp Tiered账户和v2全部SOXX门。Recorder `{summary["recorder_id"]}`，NAS SHA `{remote}`。\n'
)
print(summary['decision'],len(train),known_train,len(held),known_held,summary['status_counts'],summary['recorder_id'])


conditional_source_gate_passed_first_receipt_unqualified_no_fit 1061 1060 944 944 {'known': 2005, 'invalid_first_hour_bar': 1} 723f406c9a234790b6a96939754edd44


## 中文结论

原Stage1 10:32恢复动作的同日SOXX首小时带方向成交量源在2018–2022训练动作日历中1060/1061已知，2023–2026复用评价动作日历944/944已知。2020-03-16异常日和2023-01-03桥接日保留，未读取净动作优势标签、拟合或新账户。实际首收/修订时钟未认证；下一步先冻结与旧21项输入和实际beta0.6净恢复优势匹配的信息门。Qlib原生读回及NAS 8工件限定恢复通过。